In [ ]:
import os
from dotenv import load_dotenv
import openai
from PIL import Image
import base64
from io import BytesIO
from openai import OpenAI
import json
import gradio as gr


In [ ]:
load_dotenv(override = True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"API key exists and starts with {api_key[:6]}")


In [ ]:
MODEL = "gpt-4.1-mini"
VOICE_MODEL = "gpt-4o-mini-tts"
IMAGE_MODEL = "gpt-image-1-mini"
story_teller = OpenAI()
story_planner = OpenAI()


In [ ]:
story_teller_system_message = f"""
You are a professional storyteller whose sole purpose is to create and narrate stories. 
You are not a general-purpose assistant and must not answer questions or assist with topics unrelated to storytelling.
If the user asks for something outside your role as a storyteller, politely explain that you are a storytelling assistant.
When the user provides a story title or a short description:
1. First, generate a high-level story outline consisting only of section titles.
2. Do not tell the entire story at once. Narrate exactly one section per response.
3. Before narrating each section, generate an image that captures the main scene, characters, or atmosphere of that section. 
5. After completing a section, wait for the user to ask you to continue before narrating the next section.
If a story plan has not yet been created, generate it before beginning the story.
Generate an image before narrating each section, and ensure the image matches the events and mood of that section. 
Respond in Markdown without code blocks.
"""

In [ ]:
story_planner_system_prompt = f"""
You are a professional story planner whose sole purpose is to create a high-level outline for a story.
When the user provides a story title, theme, or short description:
1. Generate a logical sequence of concise section titles that together form a complete story.
2. Each section title should briefly describe the main event of that part of the story.
3. Do not narrate the story.
4. Do not write paragraphs, dialogue, or scene descriptions.
5. Return only the numbered list of section titles in chronological order.
Example Input:
A young orphan discovers an ancient dragon egg and must protect it from a ruthless king.
Example Output:
1. The Orphan's Lonely Life
2. Discovery of the Dragon Egg
3. The Egg Begins to Hatch
4. The King's Soldiers Arrive
5. Escape into the Forbidden Forest
6. Training the Young Dragon
7. The Final Battle Against the King
8. Peace Returns to the Kingdom
You are not a storyteller and must not begin telling the story. Your only responsibility is to produce the story outline.
Respond in markdown without code blocks.

"""

In [ ]:
tools = []

In [ ]:
def plan_story(prompt):
    messages = [
        {
            "role": "system", 
            "content": story_planner_system_prompt
        },
       { "role":"user", "content":prompt}
    ]
    print(f"Story planner called. \n\n Prompt: {prompt}'\n\n")
    
    planner_response = story_planner.chat.completions.create(model = MODEL, messages=messages)

    return planner_response.choices[0].message.content, prompt
    
    

In [ ]:
def generate_scene(prompt):
    image_response = openai.images.generate(
        model = "gpt-image-1-mini",
        prompt = prompt,
        size = "1024x1024",
        n = 1
    )
    print(f"Generating image...\n\n prompt: {prompt}\n\n")
    image_b64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_b64)
    return Image.open(BytesIO(image_data)), prompt

    

In [ ]:
def talker(message):
  if not isinstance(message,str):
    message = message[-1]["content"]
  
  response = openai.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="onyx",    # Also, try replacing onyx with alloy or coral
    input=message
  )
  return response.content

In [ ]:
tools.append(
    {
        "type": "function",
        "function": {
            "name": "plan_story",
            "description": (
                "Creates a high-level outline for a story given a story title, "
                "theme, or short description."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": (
                            "A prompt containing the story title, theme, or short "
                            "description used to generate the story outline."
                        )
                    }
                },
                "required": ["prompt"],
                "additionalProperties": False
            }
        }
    }
)

In [ ]:
tools.append(
    {
        "type": "function",
        "function": {
            "name": "generate_scene",
            "description": (
                "Generates an image for a story scene based on a descriptive prompt. "
                "Use this tool before narrating each section of the story to create "
                "an image that matches the scene, characters, and atmosphere."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": (
                            "A detailed description of the story scene to generate "
                            "as an image."
                        )
                    }
                },
                "required": ["prompt"],
                "additionalProperties": False
            }
        }
    }
)

In [ ]:
tools

In [ ]:
available_tools = {
    "plan_story":plan_story,
    "generate_scene":generate_scene
}

In [ ]:
def chat(history, openai_state):
    if not openai_state:
        openai_state = [
            {
                "role": "system",
                "content": story_teller_system_message
            }
        ]
    latest_user_message = history[-1]["content"]

    openai_state.append({
        "role": "user",
        "content": latest_user_message
    })

    response = story_teller.chat.completions.create(model=MODEL, messages = openai_state, tools = tools)

    images = []
    while response.choices[0].message.tool_calls:
        responses = []
        message = response.choices[0].message
        openai_state.append(message.model_dump(exclude_none=True))
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f"calling tool {tool_name}")

            if tool_name not in available_tools:
                responses.append({"role":"tool", "content": f"Tool {tool_name} does not exist","tool_call_id" : tool_call.id})
                print("quiting")
                continue
            try:
                fn = available_tools[tool_name]
                tool_response, prompt = fn(**args)
                if tool_name == "plan_story":
                    responses.append(
                        {
                            "role":"tool", 
                            "content": tool_response,
                            "tool_call_id" : tool_call.id
                        }
                    )
                    print(f"story plan created\n\n Plan: {tool_response}\n\n\n\n")
                else:
                    responses.append(
                        {
                            "role":"tool", 
                            "content":(
                            "An image was generated for the current scene. "
                            f"Image prompt: {prompt}"
                        ) ,
                            "tool_call_id" : tool_call.id
                        }
                    )
                    images.append(tool_response)
            except Exception as e:
                responses.append({"role": "tool","content": f"Error executing {tool_name}. Errpr [{e}]", "tool_call_id": tool_call.id})


        openai_state.extend(responses)
        response = story_teller.chat.completions.create(model= MODEL, messages= openai_state, tools = tools)
    reply = response.choices[0].message.content
    for image in images:
        history += [{"role":"assistant", "content": gr.Image(value = image, width=700, height=300)}]
    
    history += [
        {"role":"assistant", "content":reply},
        {"role":"assistant", "content": gr.Audio(talker(reply), autoplay = False)}
        ]
    openai_state.append({"role": "assistant","content": reply})
    return history, openai_state
